# Position Analysis HDF5 Overview

This notebook is the starting point for exploring the canonical `position_analysis.h5` files for the 2026-04-30 U251 live spinning-disk experiment. It autodiscovers files from the analysis root, reads their HDF5 tables, builds a compact catalog, and writes a project description summarizing what is available.

Only files matching `pos*/4_analysis/position_analysis.h5` are included. Intermediate HDF5 caches such as graphcut unary caches are intentionally ignored.

## Biological Interpretation

At a high level, these files describe how a mixed U251 glioblastoma cell population moves, packs, contacts neighboring cells, and rearranges over time in confined circular 300 um regions. Each HDF5 file corresponds to one microscope position in the live-imaging experiment.

The files do not store raw image pixels. They store downstream quantitative measurements derived from segmented and tracked cell and nucleus label images. The `cells/table` rows represent tracked cells at individual movie frames, including position, area, perimeter, bounding box, class/status, and nuclear marker summaries. The `edges/table` rows represent cell-cell contacts or adjacencies, so the tissue can be analyzed as a changing neighbor graph. The `t1_events/table` rows capture T1 transitions: local neighbor-exchange events where one cell-cell contact is lost and another is gained.

Biologically, this makes the dataset useful for studying cell density, cell shape, radial organization, contact network structure, and tissue fluidity in the WT-NLS-mCherry and VimentinKO U251 populations. Any biological interpretation should still be checked against segmentation quality and the exact meaning of `class_label` and `nls_status` before comparing cell populations.

In [1]:
from __future__ import annotations

from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import tifffile

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

ROOT = Path("/home/aruppel/Data/2026-04-30_U251-WT-NLS-mCherry_U251-VimentinKO_circle300um_live_spinning-disk/analysis")
NOTEBOOK_DIR = Path("notebooks") if Path("notebooks/position_analysis_h5_overview.ipynb").exists() else Path(".")
PROJECT_DESCRIPTION_PATH = NOTEBOOK_DIR / "position_analysis_project_description.md"

def discover_position_artifacts(root: Path = ROOT) -> list[Path]:
    """Return canonical per-position analysis artifacts, sorted by position name."""
    return sorted(root.glob("pos*/4_analysis/position_analysis.h5"))

artifact_paths = discover_position_artifacts()
print(f"Discovered {len(artifact_paths)} position_analysis.h5 files under {ROOT}")
for path in artifact_paths:
    print(path.relative_to(ROOT))

Discovered 8 position_analysis.h5 files under /home/aruppel/Data/2026-04-30_U251-WT-NLS-mCherry_U251-VimentinKO_circle300um_live_spinning-disk/analysis
pos00/4_analysis/position_analysis.h5
pos01/4_analysis/position_analysis.h5
pos02/4_analysis/position_analysis.h5
pos03/4_analysis/position_analysis.h5
pos04/4_analysis/position_analysis.h5
pos06/4_analysis/position_analysis.h5
pos07/4_analysis/position_analysis.h5
pos08/4_analysis/position_analysis.h5


## Reader Helpers

The CellFlow artifact uses HDF5 groups as column tables. The helpers below convert those groups into pandas DataFrames and carry the position label (`pos00`, `pos01`, ...) on every row so pooled analysis can group or filter by position.

In [2]:
def read_dataset(dataset: h5py.Dataset) -> np.ndarray:
    """Read a dataset, decoding HDF5 strings to Python strings."""
    if h5py.check_string_dtype(dataset.dtype) is not None:
        return dataset.asstr()[:]
    return dataset[:]


def read_table(group: h5py.Group) -> pd.DataFrame:
    """Read a CellFlow column-table HDF5 group into a DataFrame."""
    columns = {name: read_dataset(dataset) for name, dataset in group.items()}
    return pd.DataFrame(columns)


def group_inventory(h5: h5py.File) -> pd.DataFrame:
    """Return one row per dataset with path, shape, and dtype."""
    rows = []

    def visit(name: str, obj) -> None:
        if isinstance(obj, h5py.Dataset):
            rows.append({"dataset": name, "shape": obj.shape, "dtype": str(obj.dtype)})

    h5.visititems(visit)
    return pd.DataFrame(rows)


def read_artifact(path: Path, root: Path = ROOT) -> dict[str, object]:
    """Read one position_analysis.h5 artifact into tables and metadata."""
    position = path.parents[1].name
    with h5py.File(path, "r") as h5:
        cells = read_table(h5["cells/table"])
        edges = read_table(h5["edges/table"])
        t1_events = read_table(h5["t1_events/table"])
        provenance = {key: h5["provenance"].attrs[key] for key in h5["provenance"].attrs.keys()}
        inventory = group_inventory(h5)

    for table in (cells, edges, t1_events):
        table.insert(0, "position", position)

    return {
        "path": path,
        "relative_path": path.relative_to(root),
        "position": position,
        "cells": cells,
        "edges": edges,
        "t1_events": t1_events,
        "provenance": provenance,
        "inventory": inventory,
    }


artifacts = [read_artifact(path) for path in artifact_paths]
len(artifacts)

8

## Catalog and Pooled Tables

The `catalog` table has one row per HDF5 file. The `cells`, `edges`, and `t1_events` tables concatenate all discovered positions for downstream number crunching.

In [3]:
catalog_rows = []
for item in artifacts:
    cells_i = item["cells"]
    edges_i = item["edges"]
    t1_i = item["t1_events"]
    frames = sorted(cells_i["frame"].unique().tolist()) if "frame" in cells_i else []
    catalog_rows.append(
        {
            "position": item["position"],
            "path": str(item["relative_path"]),
            "n_cell_rows": len(cells_i),
            "n_unique_cell_ids": cells_i["cell_id"].nunique() if "cell_id" in cells_i else np.nan,
            "n_edge_rows": len(edges_i),
            "n_unique_edge_ids": edges_i["edge_id"].nunique() if "edge_id" in edges_i else np.nan,
            "n_t1_events": len(t1_i),
            "first_frame": min(frames) if frames else np.nan,
            "last_frame": max(frames) if frames else np.nan,
            "n_frames_with_cells": len(frames),
            "created_at": item["provenance"].get("created_at", ""),
            "cellflow_version": item["provenance"].get("cellflow_version", ""),
        }
    )

catalog = pd.DataFrame(catalog_rows).sort_values("position").reset_index(drop=True)
cells = pd.concat([item["cells"] for item in artifacts], ignore_index=True) if artifacts else pd.DataFrame()
edges = pd.concat([item["edges"] for item in artifacts], ignore_index=True) if artifacts else pd.DataFrame()
t1_events = pd.concat([item["t1_events"] for item in artifacts], ignore_index=True) if artifacts else pd.DataFrame()

catalog

,position,path,n_cell_rows,n_unique_cell_ids,n_edge_rows,n_unique_edge_ids,n_t1_events,first_frame,last_frame,n_frames_with_cells,created_at,cellflow_version
0,pos00,pos00/4_analysis/position_analysis.h5,2762,60,23152,312,112,0,47,48,2026-05-12T23:34:43.930081+00:00,0.2.0
1,pos01,pos01/4_analysis/position_analysis.h5,1559,34,12323,158,14,0,49,50,2026-05-10T08:09:29.194022+00:00,0.2.0
2,pos02,pos02/4_analysis/position_analysis.h5,2014,48,15515,236,46,0,43,44,2026-05-12T23:31:00.515592+00:00,0.2.0
3,pos03,pos03/4_analysis/position_analysis.h5,1497,31,11319,146,19,0,49,50,2026-05-12T23:33:15.695974+00:00,0.2.0
4,pos04,pos04/4_analysis/position_analysis.h5,3091,64,19065,385,184,0,49,50,2026-05-12T19:15:34.361170+00:00,0.2.0
5,pos06,pos06/4_analysis/position_analysis.h5,1210,25,9415,104,2,0,49,50,2026-05-13T22:42:35.015018+00:00,0.2.0
6,pos07,pos07/4_analysis/position_analysis.h5,2677,55,24447,317,132,0,49,50,2026-05-14T01:25:50.419304+00:00,0.2.0
7,pos08,pos08/4_analysis/position_analysis.h5,3144,66,27758,469,222,0,49,50,2026-05-14T13:16:31.564225+00:00,0.2.0


In [4]:
print("Pooled table shapes")
print(f"cells:     {cells.shape}")
print(f"edges:     {edges.shape}")
print(f"t1_events: {t1_events.shape}")

display(cells.head())
display(edges.head())
display(t1_events.head())

Pooled table shapes
cells:     (17954, 16)
edges:     (142994, 14)
t1_events: (731, 10)


,position,frame,cell_id,class_label,area,centroid_y,centroid_x,perimeter,bbox_min_y,bbox_min_x,bbox_max_y,bbox_max_x,nls_status,nls_track_median_intensity,nls_track_pixel_count,nls_track_frame_count
0,pos00,0,1,,4426.0,75.936737,346.304338,327.155375,32,285,116,399,NaN,NaN,NaN,NaN
1,pos00,0,2,,1697.0,75.147908,148.365351,206.929978,44,123,116,175,NaN,NaN,NaN,NaN
2,pos00,0,3,,3172.0,106.177491,290.428121,238.936075,79,246,145,329,NaN,NaN,NaN,NaN
3,pos00,0,4,,2194.0,107.513674,100.912489,226.492424,68,70,152,131,NaN,NaN,NaN,NaN
4,pos00,0,5,,3167.0,132.402589,417.971898,271.492424,99,374,170,460,NaN,NaN,NaN,NaN


,position,frame,edge_id,cell_a,cell_b,kind,edge_label,is_t1_frame,t1_event_id,length,midpoint_y,midpoint_x,coord_offset,coord_count
0,pos00,0,1,1,3,cell_cell,,False,-1,22.435029,92.5,323.0,0,29
1,pos00,0,2,1,5,cell_cell,,False,-1,29.970563,108.5,385.0,29,38
2,pos00,0,3,1,58,cell_cell,,False,-1,27.263456,48.5,298.0,67,35
3,pos00,0,4,2,4,cell_cell,,False,-1,31.142136,80.5,129.0,102,38
4,pos00,0,5,2,45,cell_cell,,False,-1,24.071068,105.0,131.5,140,28


,position,t1_event_id,frame,edge_id,losing_cell_a,losing_cell_b,gaining_cell_a,gaining_cell_b,location_y,location_x
0,pos00,1,0,63,19,48,11,49,210.0,388.5
1,pos00,2,0,120,54,58,52,56,73.5,224.0
2,pos00,3,1,63,11,49,19,48,211.5,387.0
3,pos00,4,1,78,26,42,24,40,320.5,98.0
4,pos00,5,1,88,29,38,27,32,369.5,145.0


## Population Identity and Marker QC

Before comparing WT-NLS-mCherry and VimentinKO behavior, check what the identity/status columns contain and whether the nuclear marker intensity separates into interpretable groups.

In [ ]:
marker_columns = [
    "position",
    "frame",
    "cell_id",
    "class_label",
    "nls_status",
    "nls_track_median_intensity",
    "nls_track_pixel_count",
    "nls_track_frame_count",
]
marker_columns = [column for column in marker_columns if column in cells.columns]

population_preview = cells[marker_columns].sort_values(["position", "frame", "cell_id"]).head(20)
population_preview

In [ ]:
identity_columns = [column for column in ["class_label", "nls_status"] if column in cells.columns]

population_counts = (
    cells.groupby(identity_columns, dropna=False)
    .agg(
        n_rows=("cell_id", "size"),
        n_cell_tracks=("cell_id", "nunique"),
        n_positions=("position", "nunique"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
    if identity_columns else pd.DataFrame()
)

population_counts

In [ ]:
marker_intensity_summary = cells["nls_track_median_intensity"].describe() if "nls_track_median_intensity" in cells.columns else pd.Series(dtype=float)

marker_by_identity = (
    cells.groupby(identity_columns, dropna=False)["nls_track_median_intensity"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .reset_index()
    .sort_values("median", ascending=False)
    if identity_columns and "nls_track_median_intensity" in cells.columns else pd.DataFrame()
)

display(marker_intensity_summary)
display(marker_by_identity)

## Population Composition Over Time

These summaries ask whether positions have comparable mixtures of `ctrl` and `vimentin_ko` cells, and whether the population fractions drift over the movie.

In [ ]:
population_by_position = (
    cells.groupby(["position", "class_label"], dropna=False, as_index=False)
    .agg(
        n_rows=("cell_id", "size"),
        n_tracks=("cell_id", "nunique"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        median_marker=("nls_track_median_intensity", "median"),
        median_area=("area", "median"),
    )
    .sort_values(["position", "class_label"])
)

population_by_position

In [ ]:
population_by_frame = (
    cells.groupby(["position", "frame", "class_label"], dropna=False, as_index=False)
    .agg(
        n_cells=("cell_id", "nunique"),
        median_marker=("nls_track_median_intensity", "median"),
        median_area=("area", "median"),
    )
    .sort_values(["position", "frame", "class_label"])
)

frame_totals = (
    population_by_frame.groupby(["position", "frame"], as_index=False)["n_cells"]
    .sum()
    .rename(columns={"n_cells": "total_cells"})
)

population_fraction_by_frame = population_by_frame.merge(frame_totals, on=["position", "frame"], how="left")
population_fraction_by_frame["fraction_of_frame"] = population_fraction_by_frame["n_cells"] / population_fraction_by_frame["total_cells"]

population_fraction_by_frame.head(20)

In [ ]:
population_fraction_summary = (
    population_fraction_by_frame.groupby(["position", "class_label"], dropna=False, as_index=False)
    .agg(
        mean_fraction=("fraction_of_frame", "mean"),
        min_fraction=("fraction_of_frame", "min"),
        max_fraction=("fraction_of_frame", "max"),
        mean_cells_per_frame=("n_cells", "mean"),
        min_cells_per_frame=("n_cells", "min"),
        max_cells_per_frame=("n_cells", "max"),
    )
    .sort_values(["position", "class_label"])
)

population_fraction_summary

## Cell Morphology By Population

These summaries compare basic cell morphology between `ctrl` and `vimentin_ko`: area, perimeter, and a simple shape index defined as `perimeter / sqrt(area)`. The position-level table is the safer first read because it shows whether the pooled difference is consistent across fields of view.

In [ ]:
cells["shape_index"] = cells["perimeter"] / np.sqrt(cells["area"])

morphology_by_population = (
    cells.groupby("class_label", dropna=False, as_index=False)
    .agg(
        n_rows=("cell_id", "size"),
        n_tracks=("cell_id", "nunique"),
        median_area=("area", "median"),
        mean_area=("area", "mean"),
        median_perimeter=("perimeter", "median"),
        mean_perimeter=("perimeter", "mean"),
        median_shape_index=("shape_index", "median"),
        mean_shape_index=("shape_index", "mean"),
    )
    .sort_values("class_label")
)

morphology_by_population

In [ ]:
morphology_by_position = (
    cells.groupby(["position", "class_label"], dropna=False, as_index=False)
    .agg(
        n_rows=("cell_id", "size"),
        n_tracks=("cell_id", "nunique"),
        median_area=("area", "median"),
        mean_area=("area", "mean"),
        median_perimeter=("perimeter", "median"),
        mean_perimeter=("perimeter", "mean"),
        median_shape_index=("shape_index", "median"),
        mean_shape_index=("shape_index", "mean"),
    )
    .sort_values(["position", "class_label"])
)

morphology_by_position

## Cell-Cell Contact Composition

Annotate each cell-cell edge with the population identity of both neighboring cells. This lets us ask whether contacts are mostly `ctrl-ctrl`, `ctrl-vimentin_ko`, or `vimentin_ko-vimentin_ko`, and whether contact length differs by pair type.

In [ ]:
cell_identity = cells[["position", "frame", "cell_id", "class_label", "nls_status"]].drop_duplicates(
    ["position", "frame", "cell_id"]
)
cell_cell_edges = edges[edges["kind"].eq("cell_cell")].copy()

edge_kind_summary = (
    edges.groupby("kind", dropna=False, as_index=False)
    .agg(n_edges=("edge_id", "size"), median_length=("length", "median"), mean_length=("length", "mean"))
    .sort_values("n_edges", ascending=False)
)

edge_classes = (
    cell_cell_edges.merge(
        cell_identity.rename(columns={"cell_id": "cell_a", "class_label": "class_a", "nls_status": "nls_status_a"}),
        on=["position", "frame", "cell_a"],
        how="left",
    )
    .merge(
        cell_identity.rename(columns={"cell_id": "cell_b", "class_label": "class_b", "nls_status": "nls_status_b"}),
        on=["position", "frame", "cell_b"],
        how="left",
    )
)

edge_classes["pair_type"] = edge_classes.apply(
    lambda row: "-".join(sorted([str(row["class_a"]), str(row["class_b"])])),
    axis=1,
)

display(edge_kind_summary)
edge_classes[["position", "frame", "edge_id", "cell_a", "cell_b", "class_a", "class_b", "pair_type", "length"]].head(20)

In [ ]:
contact_pair_summary = (
    edge_classes.groupby("pair_type", dropna=False, as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        n_positions=("position", "nunique"),
        median_length=("length", "median"),
        mean_length=("length", "mean"),
        t1_marked_contacts=("is_t1_frame", "sum"),
    )
    .sort_values("n_contacts", ascending=False)
)
contact_pair_summary["fraction_of_contacts"] = contact_pair_summary["n_contacts"] / contact_pair_summary["n_contacts"].sum()

contact_pair_summary

In [ ]:
contact_pair_by_position = (
    edge_classes.groupby(["position", "pair_type"], dropna=False, as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        median_length=("length", "median"),
        mean_length=("length", "mean"),
        t1_marked_contacts=("is_t1_frame", "sum"),
    )
    .sort_values(["position", "pair_type"])
)

position_contact_totals = contact_pair_by_position.groupby("position", as_index=False)["n_contacts"].sum().rename(
    columns={"n_contacts": "position_contacts"}
)
contact_pair_by_position = contact_pair_by_position.merge(position_contact_totals, on="position", how="left")
contact_pair_by_position["fraction_of_position_contacts"] = contact_pair_by_position["n_contacts"] / contact_pair_by_position["position_contacts"]

contact_pair_by_position

In [ ]:
contact_pair_by_frame = (
    edge_classes.groupby(["position", "frame", "pair_type"], dropna=False, as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        median_length=("length", "median"),
        mean_length=("length", "mean"),
    )
    .sort_values(["position", "frame", "pair_type"])
)

contact_pair_by_frame.head(20)

## Contact Mixing: Observed Versus Expected

Compare observed contact pair fractions to random-mixing expectations from the population fractions in the same position and frame. For two classes with fractions `p_ctrl` and `p_ko`, the random expectation is `p_ctrl ** 2` for `ctrl-ctrl`, `2 * p_ctrl * p_ko` for mixed contacts, and `p_ko ** 2` for `vimentin_ko-vimentin_ko`.

In [ ]:
population_fraction_wide = (
    population_fraction_by_frame.pivot_table(
        index=["position", "frame"],
        columns="class_label",
        values="fraction_of_frame",
        fill_value=0,
    )
    .reset_index()
)

for class_label in ["ctrl", "vimentin_ko"]:
    if class_label not in population_fraction_wide.columns:
        population_fraction_wide[class_label] = 0.0

expected_contact_fraction_by_frame = pd.concat(
    [
        population_fraction_wide.assign(
            pair_type="ctrl-ctrl",
            expected_fraction=population_fraction_wide["ctrl"] ** 2,
        )[["position", "frame", "pair_type", "expected_fraction"]],
        population_fraction_wide.assign(
            pair_type="ctrl-vimentin_ko",
            expected_fraction=2 * population_fraction_wide["ctrl"] * population_fraction_wide["vimentin_ko"],
        )[["position", "frame", "pair_type", "expected_fraction"]],
        population_fraction_wide.assign(
            pair_type="vimentin_ko-vimentin_ko",
            expected_fraction=population_fraction_wide["vimentin_ko"] ** 2,
        )[["position", "frame", "pair_type", "expected_fraction"]],
    ],
    ignore_index=True,
)

expected_contact_fraction_by_frame.head()

In [ ]:
contact_totals_by_frame = (
    contact_pair_by_frame.groupby(["position", "frame"], as_index=False)["n_contacts"]
    .sum()
    .rename(columns={"n_contacts": "total_contacts"})
)

observed_contact_fraction_by_frame = contact_pair_by_frame.merge(
    contact_totals_by_frame,
    on=["position", "frame"],
    how="left",
)
observed_contact_fraction_by_frame["observed_fraction"] = (
    observed_contact_fraction_by_frame["n_contacts"] / observed_contact_fraction_by_frame["total_contacts"]
)

contact_mixing_by_frame = observed_contact_fraction_by_frame.merge(
    expected_contact_fraction_by_frame,
    on=["position", "frame", "pair_type"],
    how="left",
)
contact_mixing_by_frame["observed_minus_expected"] = (
    contact_mixing_by_frame["observed_fraction"] - contact_mixing_by_frame["expected_fraction"]
)
contact_mixing_by_frame["observed_over_expected"] = (
    contact_mixing_by_frame["observed_fraction"] / contact_mixing_by_frame["expected_fraction"].replace(0, np.nan)
)

contact_mixing_by_frame.head(20)

In [ ]:
contact_mixing_summary = (
    contact_mixing_by_frame.groupby(["position", "pair_type"], as_index=False)
    .agg(
        mean_observed_fraction=("observed_fraction", "mean"),
        mean_expected_fraction=("expected_fraction", "mean"),
        mean_observed_minus_expected=("observed_minus_expected", "mean"),
        mean_observed_over_expected=("observed_over_expected", "mean"),
        n_frames=("frame", "nunique"),
    )
    .sort_values(["position", "pair_type"])
)

contact_mixing_summary

## T1 Rearrangements By Contact Pair Type

These summaries ask whether contact instability differs by population pairing. The rate is normalized by the number of observed cell-cell contacts of each pair type, so mixed contacts are compared to same-type contacts on their own exposure.

In [ ]:
t1_contact_rate_by_pair = (
    edge_classes.groupby("pair_type", as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        n_t1_contacts=("is_t1_frame", "sum"),
        t1_contact_rate=("is_t1_frame", "mean"),
        median_length=("length", "median"),
        mean_length=("length", "mean"),
    )
    .sort_values("t1_contact_rate", ascending=False)
)

t1_contact_rate_by_pair

In [ ]:
t1_contact_rate_by_position = (
    edge_classes.groupby(["position", "pair_type"], as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        n_t1_contacts=("is_t1_frame", "sum"),
        t1_contact_rate=("is_t1_frame", "mean"),
        median_length=("length", "median"),
        mean_length=("length", "mean"),
    )
    .sort_values(["position", "pair_type"])
)

t1_contact_rate_by_position

In [ ]:
t1_contact_fraction_by_frame = (
    edge_classes.groupby(["position", "frame", "pair_type"], as_index=False)
    .agg(
        n_contacts=("edge_id", "size"),
        n_t1_contacts=("is_t1_frame", "sum"),
    )
    .sort_values(["position", "frame", "pair_type"])
)
t1_contact_fraction_by_frame["t1_contact_rate"] = (
    t1_contact_fraction_by_frame["n_t1_contacts"] / t1_contact_fraction_by_frame["n_contacts"]
)

t1_contact_fraction_by_frame.head(20)

## Nucleus-Based Migration Speed

Migration is computed from tracked nucleus centroids, not cell-mask centroids. The `position_analysis.h5` files store the path to each `2_nucleus/tracked_labels.tif` in provenance, so this section reads those label stacks, extracts nucleus centroids by `cell_id`/track, and computes consecutive-frame centroid displacement in pixels per frame.

In [ ]:
def nucleus_centroids_from_labels(labels: np.ndarray) -> pd.DataFrame:
    labels = np.asarray(labels)
    if labels.ndim == 2:
        labels = labels[np.newaxis, ...]
    if labels.ndim > 3:
        labels = np.squeeze(labels)
    if labels.ndim != 3:
        raise ValueError(f"Expected time-first 2D/3D nucleus labels, got shape {labels.shape}")

    rows = []
    for frame_idx, frame in enumerate(labels):
        flat = frame.ravel()
        order = np.argsort(flat, kind="stable")
        sorted_ids = flat[order]

        change = np.empty(len(sorted_ids), dtype=bool)
        change[0] = True
        np.not_equal(sorted_ids[1:], sorted_ids[:-1], out=change[1:])
        starts = np.flatnonzero(change)

        ends = np.empty_like(starts)
        ends[:-1] = starts[1:]
        ends[-1] = len(sorted_ids)

        ys_all, xs_all = np.divmod(order, frame.shape[1])
        for start, end in zip(starts, ends):
            cell_id = int(sorted_ids[start])
            if cell_id == 0:
                continue
            rows.append(
                {
                    "frame": frame_idx,
                    "cell_id": cell_id,
                    "nucleus_centroid_y": float(ys_all[start:end].mean()),
                    "nucleus_centroid_x": float(xs_all[start:end].mean()),
                    "nucleus_area": int(end - start),
                }
            )
    return pd.DataFrame(rows)


def read_nucleus_centroids_for_artifact(item: dict) -> pd.DataFrame:
    nucleus_path = Path(item["provenance"]["nucleus_tracked_labels_path"])
    centroids = nucleus_centroids_from_labels(tifffile.imread(nucleus_path))
    centroids.insert(0, "position", item["position"])
    centroids.insert(1, "nucleus_labels_path", str(nucleus_path))
    return centroids


nucleus_centroids = pd.concat(
    [read_nucleus_centroids_for_artifact(item) for item in artifacts],
    ignore_index=True,
)

nucleus_centroids.head()

In [ ]:
nucleus_identity = cells[["position", "frame", "cell_id", "class_label", "nls_status"]].drop_duplicates(
    ["position", "frame", "cell_id"]
)

nucleus_motion = (
    nucleus_centroids.merge(nucleus_identity, on=["position", "frame", "cell_id"], how="left")
    .sort_values(["position", "cell_id", "frame"])
    .reset_index(drop=True)
)

track_groups = nucleus_motion.groupby(["position", "cell_id"])
nucleus_motion["dt"] = track_groups["frame"].diff()
nucleus_motion["dy"] = track_groups["nucleus_centroid_y"].diff()
nucleus_motion["dx"] = track_groups["nucleus_centroid_x"].diff()
nucleus_motion["step_distance_px"] = np.sqrt(nucleus_motion["dy"] ** 2 + nucleus_motion["dx"] ** 2)
nucleus_motion["speed_px_per_frame"] = nucleus_motion["step_distance_px"] / nucleus_motion["dt"]

nucleus_speed_steps = nucleus_motion[nucleus_motion["dt"].eq(1)].copy()

track_gap_qc = (
    nucleus_motion[nucleus_motion["dt"].notna()]
    .groupby("dt", as_index=False)
    .size()
    .rename(columns={"size": "n_steps"})
    .sort_values("dt")
)

display(track_gap_qc)
nucleus_speed_steps.head()

In [ ]:
nucleus_migration_by_population = (
    nucleus_speed_steps.groupby("class_label", dropna=False, as_index=False)
    .agg(
        n_steps=("speed_px_per_frame", "size"),
        n_tracks=("cell_id", "nunique"),
        median_speed_px_per_frame=("speed_px_per_frame", "median"),
        mean_speed_px_per_frame=("speed_px_per_frame", "mean"),
        median_step_distance_px=("step_distance_px", "median"),
        mean_step_distance_px=("step_distance_px", "mean"),
    )
    .sort_values("class_label")
)

nucleus_migration_by_population

In [ ]:
nucleus_migration_by_position = (
    nucleus_speed_steps.groupby(["position", "class_label"], dropna=False, as_index=False)
    .agg(
        n_steps=("speed_px_per_frame", "size"),
        n_tracks=("cell_id", "nunique"),
        median_speed_px_per_frame=("speed_px_per_frame", "median"),
        mean_speed_px_per_frame=("speed_px_per_frame", "mean"),
        median_step_distance_px=("step_distance_px", "median"),
        mean_step_distance_px=("step_distance_px", "mean"),
    )
    .sort_values(["position", "class_label"])
)

nucleus_migration_by_position

## Nucleus Speed Versus Tissue Density

Join each nucleus displacement step to field-level density measured at the step's ending frame. This first-pass density is per position/frame: number of segmented cells, total segmented cell area, and median cell area. Later analyses can replace this with local density around each nucleus.

In [ ]:
density_by_frame = (
    cells.groupby(["position", "frame"], as_index=False)
    .agg(
        n_cells=("cell_id", "nunique"),
        total_cell_area=("area", "sum"),
        median_cell_area=("area", "median"),
        mean_cell_area=("area", "mean"),
    )
    .sort_values(["position", "frame"])
)

speed_density = nucleus_speed_steps.merge(
    density_by_frame,
    on=["position", "frame"],
    how="left",
)

speed_density.head()

In [ ]:
speed_density = speed_density.copy()
speed_density["n_cells_bin"] = pd.qcut(speed_density["n_cells"], q=4, duplicates="drop")
speed_density["total_cell_area_bin"] = pd.qcut(speed_density["total_cell_area"], q=4, duplicates="drop")
speed_density["density_bin"] = speed_density["n_cells_bin"]

speed_by_density_bin = (
    speed_density.groupby(["class_label", "density_bin"], observed=True, as_index=False)
    .agg(
        n_steps=("speed_px_per_frame", "size"),
        n_tracks=("cell_id", "nunique"),
        median_speed_px_per_frame=("speed_px_per_frame", "median"),
        mean_speed_px_per_frame=("speed_px_per_frame", "mean"),
        median_n_cells=("n_cells", "median"),
        median_total_cell_area=("total_cell_area", "median"),
    )
    .sort_values(["class_label", "density_bin"])
)

speed_by_density_bin

In [ ]:
def corr_or_nan(frame: pd.DataFrame, x: str, y: str) -> float:
    if frame[x].nunique(dropna=True) < 2 or frame[y].nunique(dropna=True) < 2:
        return np.nan
    return float(frame[[x, y]].corr().loc[x, y])


speed_density_correlation = (
    speed_density.groupby(["position", "class_label"], dropna=False)
    .apply(
        lambda group: pd.Series(
            {
                "n_steps": len(group),
                "n_tracks": group["cell_id"].nunique(),
                "corr_speed_n_cells": corr_or_nan(group, "speed_px_per_frame", "n_cells"),
                "corr_speed_total_cell_area": corr_or_nan(group, "speed_px_per_frame", "total_cell_area"),
                "median_speed_px_per_frame": group["speed_px_per_frame"].median(),
                "median_n_cells": group["n_cells"].median(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values(["position", "class_label"])
)

speed_density_correlation

## Radial Organization and Radial Migration

Estimate a per-position assay center from the nucleus centroid cloud, then ask whether populations occupy different radial zones and whether motion is biased outward or inward. Radius is normalized within each position by the 95th percentile nucleus radius, so values are comparable across fields even if the exact circle center/radius is approximate.

In [ ]:
position_radial_reference = (
    nucleus_centroids.groupby("position", as_index=False)
    .agg(
        center_y=("nucleus_centroid_y", "median"),
        center_x=("nucleus_centroid_x", "median"),
    )
)

nucleus_radial_positions = nucleus_centroids.merge(position_radial_reference, on="position", how="left")
nucleus_radial_positions["radius_px"] = np.sqrt(
    (nucleus_radial_positions["nucleus_centroid_y"] - nucleus_radial_positions["center_y"]) ** 2
    + (nucleus_radial_positions["nucleus_centroid_x"] - nucleus_radial_positions["center_x"]) ** 2
)

radius_scale = (
    nucleus_radial_positions.groupby("position")["radius_px"]
    .quantile(0.95)
    .rename("radius_p95_px")
    .reset_index()
)
position_radial_reference = position_radial_reference.merge(radius_scale, on="position", how="left")
nucleus_radial_positions = nucleus_radial_positions.merge(radius_scale, on="position", how="left")
nucleus_radial_positions["radius_norm"] = nucleus_radial_positions["radius_px"] / nucleus_radial_positions["radius_p95_px"]

nucleus_radial_positions = nucleus_radial_positions.merge(
    nucleus_identity,
    on=["position", "frame", "cell_id"],
    how="left",
)

position_radial_reference

In [ ]:
radial_position_by_population = (
    nucleus_radial_positions.groupby(["position", "class_label"], dropna=False, as_index=False)
    .agg(
        n_rows=("cell_id", "size"),
        n_tracks=("cell_id", "nunique"),
        median_radius_norm=("radius_norm", "median"),
        mean_radius_norm=("radius_norm", "mean"),
        q25_radius_norm=("radius_norm", lambda s: s.quantile(0.25)),
        q75_radius_norm=("radius_norm", lambda s: s.quantile(0.75)),
    )
    .sort_values(["position", "class_label"])
)

radial_position_by_population

In [ ]:
radial_motion_steps = nucleus_speed_steps.merge(
    position_radial_reference,
    on="position",
    how="left",
).sort_values(["position", "cell_id", "frame"]).reset_index(drop=True)

radial_motion_steps["prev_nucleus_centroid_y"] = radial_motion_steps["nucleus_centroid_y"] - radial_motion_steps["dy"]
radial_motion_steps["prev_nucleus_centroid_x"] = radial_motion_steps["nucleus_centroid_x"] - radial_motion_steps["dx"]
radial_motion_steps["prev_radius_px"] = np.sqrt(
    (radial_motion_steps["prev_nucleus_centroid_y"] - radial_motion_steps["center_y"]) ** 2
    + (radial_motion_steps["prev_nucleus_centroid_x"] - radial_motion_steps["center_x"]) ** 2
)
radial_motion_steps["prev_radius_norm"] = radial_motion_steps["prev_radius_px"] / radial_motion_steps["radius_p95_px"]
radial_motion_steps["radius_px"] = np.sqrt(
    (radial_motion_steps["nucleus_centroid_y"] - radial_motion_steps["center_y"]) ** 2
    + (radial_motion_steps["nucleus_centroid_x"] - radial_motion_steps["center_x"]) ** 2
)
radial_motion_steps["radius_norm"] = radial_motion_steps["radius_px"] / radial_motion_steps["radius_p95_px"]

nonzero_radius = radial_motion_steps["prev_radius_px"].replace(0, np.nan)
radial_motion_steps["radial_unit_y"] = (radial_motion_steps["prev_nucleus_centroid_y"] - radial_motion_steps["center_y"]) / nonzero_radius
radial_motion_steps["radial_unit_x"] = (radial_motion_steps["prev_nucleus_centroid_x"] - radial_motion_steps["center_x"]) / nonzero_radius
radial_motion_steps["radial_velocity_px_per_frame"] = (
    radial_motion_steps["dy"] * radial_motion_steps["radial_unit_y"]
    + radial_motion_steps["dx"] * radial_motion_steps["radial_unit_x"]
) / radial_motion_steps["dt"]
radial_motion_steps["tangential_speed_px_per_frame"] = np.sqrt(
    np.maximum(radial_motion_steps["speed_px_per_frame"] ** 2 - radial_motion_steps["radial_velocity_px_per_frame"] ** 2, 0)
)
radial_motion_steps["radial_direction"] = np.select(
    [radial_motion_steps["radial_velocity_px_per_frame"] > 0, radial_motion_steps["radial_velocity_px_per_frame"] < 0],
    ["outward", "inward"],
    default="zero",
)

radial_motion_steps.head()

In [ ]:
radial_motion_by_population = (
    radial_motion_steps.groupby("class_label", dropna=False, as_index=False)
    .agg(
        n_steps=("speed_px_per_frame", "size"),
        n_tracks=("cell_id", "nunique"),
        median_speed_px_per_frame=("speed_px_per_frame", "median"),
        median_radial_velocity_px_per_frame=("radial_velocity_px_per_frame", "median"),
        mean_radial_velocity_px_per_frame=("radial_velocity_px_per_frame", "mean"),
        median_tangential_speed_px_per_frame=("tangential_speed_px_per_frame", "median"),
        outward_fraction=("radial_direction", lambda s: (s == "outward").mean()),
    )
    .sort_values("class_label")
)

radial_motion_by_population

In [ ]:
radial_motion_steps = radial_motion_steps.copy()
radial_motion_steps["radius_bin"] = pd.cut(
    radial_motion_steps["prev_radius_norm"],
    bins=[0, 0.33, 0.66, 1.0, np.inf],
    labels=["inner", "middle", "outer", "beyond_p95"],
    include_lowest=True,
)

radial_motion_by_radius_bin = (
    radial_motion_steps.groupby(["class_label", "radius_bin"], observed=True, dropna=False, as_index=False)
    .agg(
        n_steps=("speed_px_per_frame", "size"),
        n_tracks=("cell_id", "nunique"),
        median_speed_px_per_frame=("speed_px_per_frame", "median"),
        median_radial_velocity_px_per_frame=("radial_velocity_px_per_frame", "median"),
        mean_radial_velocity_px_per_frame=("radial_velocity_px_per_frame", "mean"),
        outward_fraction=("radial_direction", lambda s: (s == "outward").mean()),
    )
    .sort_values(["class_label", "radius_bin"])
)

radial_motion_by_radius_bin

## HDF5 Schema Inventory

This shows the dataset paths, shapes, and dtypes found in each file. It is useful for checking whether newly added positions have the same schema as earlier positions.

In [5]:
inventory = pd.concat(
    [item["inventory"].assign(position=item["position"], path=str(item["relative_path"])) for item in artifacts],
    ignore_index=True,
) if artifacts else pd.DataFrame(columns=["dataset", "shape", "dtype", "position", "path"])

schema_summary = (
    inventory.groupby(["dataset", "dtype"], dropna=False)
    .agg(
        n_positions=("position", "nunique"),
        example_shape=("shape", "first"),
        positions=("position", lambda s: ", ".join(sorted(s.unique()))),
    )
    .reset_index()
    .sort_values("dataset")
)

schema_summary

,dataset,dtype,n_positions,example_shape,positions
0,cells/table/area,float64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
1,cells/table/bbox_max_x,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
2,cells/table/bbox_max_y,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
3,cells/table/bbox_min_x,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
4,cells/table/bbox_min_y,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
5,cells/table/cell_id,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
6,cells/table/centroid_x,float64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
7,cells/table/centroid_y,float64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
8,cells/table/class_label,object,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"
9,cells/table/frame,int64,8,"(2762,)","pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08"


## First Aggregate Checks

These are intentionally simple: counts by position and by frame. They are here as a sanity check before deeper analysis.

In [6]:
cells_per_frame = (
    cells.groupby(["position", "frame"], as_index=False)
    .agg(n_cells=("cell_id", "nunique"), total_cell_area=("area", "sum"))
    if not cells.empty else pd.DataFrame()
)

edges_per_frame = (
    edges.groupby(["position", "frame"], as_index=False)
    .agg(n_edges=("edge_id", "nunique"), mean_edge_length=("length", "mean"))
    if not edges.empty else pd.DataFrame()
)

display(cells_per_frame.head())
display(edges_per_frame.head())

,position,frame,n_cells,total_cell_area
0,pos00,0,56,141787.0
1,pos00,1,56,141167.0
2,pos00,2,57,139505.0
3,pos00,3,58,139023.0
4,pos00,4,58,140438.0


,position,frame,n_edges,mean_edge_length
0,pos00,0,172,19.427911
1,pos00,1,167,19.183750
2,pos00,2,170,16.645427
3,pos00,3,177,17.336942
4,pos00,4,176,16.643222


## Write Project Description

The next cell writes a Markdown description next to this notebook. Re-run it after new `position_analysis.h5` files arrive to refresh the counts and schema notes.

In [7]:
def bullet_list(values) -> str:
    return "\n".join(f"- `{value}`" for value in values)


def describe_columns(name: str, table: pd.DataFrame) -> str:
    if table.empty:
        return f"### {name}\n\nNo rows were loaded.\n"
    columns = [column for column in table.columns if column != "position"]
    return f"### {name}\n\nRows: {len(table):,}\n\nColumns:\n{bullet_list(columns)}\n"


description = f"""# Position Analysis HDF5 Project Description

Analysis root: `{ROOT}`

This project explores CellFlow `position_analysis.h5` files. Each file is a canonical per-position artifact built from tracked cell and nucleus label images. The artifact stores tabular cell measurements, cell-cell edge/contact measurements, T1 transition records, edge coordinate arrays, and provenance attributes.

The notebook discovers files with this pattern:

```text
pos*/4_analysis/position_analysis.h5
```

It intentionally excludes other `.h5` files such as graphcut or ICM unary caches.

## Current Discovery

Discovered position artifacts: {len(catalog):,}

Positions: {', '.join(catalog['position'].tolist()) if not catalog.empty else 'none'}

Total cell rows: {int(catalog['n_cell_rows'].sum()) if not catalog.empty else 0:,}

Total edge rows: {int(catalog['n_edge_rows'].sum()) if not catalog.empty else 0:,}

Total T1 events: {int(catalog['n_t1_events'].sum()) if not catalog.empty else 0:,}

## Tables

{describe_columns('cells/table', cells)}

{describe_columns('edges/table', edges)}

{describe_columns('t1_events/table', t1_events)}

## Provenance

Each file has a `provenance` group with attributes such as source position path, tracked cell labels path, tracked nucleus labels path, edge extraction parameters, creation timestamp, and CellFlow version.

## DataFrames Created By The Notebook

- `catalog`: one row per discovered HDF5 file
- `cells`: pooled `cells/table` rows across positions
- `edges`: pooled `edges/table` rows across positions
- `t1_events`: pooled `t1_events/table` rows across positions
- `inventory`: one row per HDF5 dataset path per position
- `schema_summary`: compact schema inventory across positions
- `cells_per_frame`: cell counts and total cell area by position/frame
- `edges_per_frame`: edge counts and mean edge length by position/frame
"""

PROJECT_DESCRIPTION_PATH.write_text(description)
print(f"Wrote {PROJECT_DESCRIPTION_PATH.resolve()}")
print(description)

Wrote /home/aruppel/Projects/CellFlow/notebooks/position_analysis_project_description.md
# Position Analysis HDF5 Project Description

Analysis root: `/home/aruppel/Data/2026-04-30_U251-WT-NLS-mCherry_U251-VimentinKO_circle300um_live_spinning-disk/analysis`

This project explores CellFlow `position_analysis.h5` files. Each file is a canonical per-position artifact built from tracked cell and nucleus label images. The artifact stores tabular cell measurements, cell-cell edge/contact measurements, T1 transition records, edge coordinate arrays, and provenance attributes.

The notebook discovers files with this pattern:

```text
pos*/4_analysis/position_analysis.h5
```

It intentionally excludes other `.h5` files such as graphcut or ICM unary caches.

## Current Discovery

Discovered position artifacts: 8

Positions: pos00, pos01, pos02, pos03, pos04, pos06, pos07, pos08

Total cell rows: 17,954

Total edge rows: 142,994

Total T1 events: 731

## Tables

### cells/table

Rows: 17,954

Colu